# Panorama Stitching (SIFT + RANSAC Homography + Feathered Blending)

Stitches three overlapping images (left, center, right) into one panorama using the center image as reference.

**Pipeline:** grayscale → SIFT keypoints → BFMatcher + Lowe's ratio test → RANSAC homography → warp onto common canvas → distance-transform blending → crop black borders.

Place `left.jpeg`, `center.jpeg`, `right.jpeg` in the same folder as this notebook (or upload them in Colab).

## Part 1 — Imports & Configuration

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

MIN_MATCH_COUNT = 20
RANSAC_REPROJ_THRESHOLD = 3.0

## Part 2 — Load Input Images

In [ ]:
img_left   = cv2.imread("left.jpeg")
img_center = cv2.imread("center.jpeg")
img_right  = cv2.imread("right.jpeg")

if any(img is None for img in [img_left, img_center, img_right]):
    raise FileNotFoundError("One or more images could not be loaded (need left.jpeg, center.jpeg, right.jpeg)")

print("Images loaded successfully")

# Convert BGR (OpenCV) to RGB for Matplotlib
img_left_rgb   = cv2.cvtColor(img_left, cv2.COLOR_BGR2RGB)
img_center_rgb = cv2.cvtColor(img_center, cv2.COLOR_BGR2RGB)
img_right_rgb  = cv2.cvtColor(img_right, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(15,5))
for i, (im, t) in enumerate([(img_left_rgb, "Left Image"),
                             (img_center_rgb, "Center Image (Reference)"),
                             (img_right_rgb, "Right Image")], 1):
    plt.subplot(1,3,i)
    plt.imshow(im)
    plt.title(t)
    plt.axis("off")
plt.show()

## Part 3 — Convert to Grayscale

In [ ]:
gray_left   = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
gray_center = cv2.cvtColor(img_center, cv2.COLOR_BGR2GRAY)
gray_right  = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(15,5))
for i, (im, t) in enumerate([(gray_left, "Left - Grayscale"),
                             (gray_center, "Center - Grayscale"),
                             (gray_right, "Right - Grayscale")], 1):
    plt.subplot(1,3,i)
    plt.imshow(im, cmap="gray")
    plt.title(t)
    plt.axis("off")
plt.show()

## Part 4 — SIFT Feature Extraction

In [ ]:
sift = cv2.SIFT_create(nfeatures=8000)

kp_l, des_l = sift.detectAndCompute(gray_left, None)
kp_c, des_c = sift.detectAndCompute(gray_center, None)
kp_r, des_r = sift.detectAndCompute(gray_right, None)

print("Keypoints:")
print("Left:", len(kp_l))
print("Center:", len(kp_c))
print("Right:", len(kp_r))

flag = cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
img_l_kp = cv2.drawKeypoints(img_left, kp_l, None, flags=flag)
img_c_kp = cv2.drawKeypoints(img_center, kp_c, None, flags=flag)
img_r_kp = cv2.drawKeypoints(img_right, kp_r, None, flags=flag)

plt.figure(figsize=(15,5))
for i, (im, t) in enumerate([(img_l_kp, "Left - SIFT Keypoints"),
                             (img_c_kp, "Center - SIFT Keypoints"),
                             (img_r_kp, "Right - SIFT Keypoints")], 1):
    plt.subplot(1,3,i)
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    plt.title(t)
    plt.axis("off")
plt.show()

## Part 5 — Homography Function (with RANSAC)

In [ ]:
def compute_homography(kp_src, des_src, kp_dst, des_dst):
    matcher = cv2.BFMatcher(cv2.NORM_L2)
    matches = matcher.knnMatch(des_src, des_dst, k=2)

    good = [m for m,n in matches if m.distance < 0.75*n.distance]

    print("Good matches:", len(good))

    if len(good) < MIN_MATCH_COUNT:
        print("Not enough matches to compute homography")
        return None, None

    src_pts = np.float32([kp_src[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst_pts = np.float32([kp_dst[m.trainIdx].pt for m in good]).reshape(-1,1,2)

    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, RANSAC_REPROJ_THRESHOLD)

    return H, good

## Part 6 — Compute Transformations

In [ ]:
H_left_to_center, matches_lc  = compute_homography(kp_l, des_l, kp_c, des_c)
H_right_to_center, matches_rc = compute_homography(kp_r, des_r, kp_c, des_c)

if H_left_to_center is None or H_right_to_center is None:
    raise ValueError("Homography failed — try images with more overlap")
else:
    print("Homographies calculated successfully")

## Part 7 — Naive Stitching (Without Blending)

In [ ]:
def corners_of(img):
    h, w = img.shape[:2]
    return np.float32([[0,0],[0,h],[w,h],[w,0]]).reshape(-1,1,2)

corners_l = corners_of(img_left)
corners_r = corners_of(img_right)
corners_c = corners_of(img_center)

# Transform left and right corners into center image coordinates
proj_l = cv2.perspectiveTransform(corners_l, H_left_to_center)
proj_r = cv2.perspectiveTransform(corners_r, H_right_to_center)

# Panorama canvas size
all_corners = np.concatenate((proj_l, proj_r, corners_c), axis=0)
xmin, ymin = np.int32(all_corners.min(axis=0).ravel())
xmax, ymax = np.int32(all_corners.max(axis=0).ravel())

# Translation to move panorama into positive coordinate space
T = np.array([[1,0,-xmin],[0,1,-ymin],[0,0,1]], dtype=np.float64)

size = (xmax-xmin, ymax-ymin)
naive       = cv2.warpPerspective(img_center, T, size)
naive_left  = cv2.warpPerspective(img_left,  T @ H_left_to_center,  size)
naive_right = cv2.warpPerspective(img_right, T @ H_right_to_center, size)

# Directly overlay warped images (no blending)
mask = naive_left > 0
naive[mask] = naive_left[mask]
mask = naive_right > 0
naive[mask] = naive_right[mask]

plt.figure(figsize=(15,6))
plt.imshow(cv2.cvtColor(naive, cv2.COLOR_BGR2RGB))
plt.title("Naive Panorama (Without Blending)")
plt.axis("off")
plt.show()

## Part 8 — Blending Function (Seam Removal + Exposure)

Distance-transform feathering: each pixel is weighted by its distance from the edge of its image's valid region, so seams fade smoothly across the overlap.

In [ ]:
def blend_images(img1, img2):

    # Canvas big enough to hold both images
    h = max(img1.shape[0], img2.shape[0])
    w = max(img1.shape[1], img2.shape[1])

    canvas1 = np.zeros((h, w, 3), dtype=np.float32)
    canvas2 = np.zeros((h, w, 3), dtype=np.float32)

    canvas1[:img1.shape[0], :img1.shape[1]] = img1
    canvas2[:img2.shape[0], :img2.shape[1]] = img2

    # Masks for valid image areas
    mask1 = (canvas1.sum(axis=2) > 0).astype(np.uint8)
    mask2 = (canvas2.sum(axis=2) > 0).astype(np.uint8)

    # Distance from edge for each mask
    dist1 = cv2.distanceTransform(mask1, cv2.DIST_L2, 3)
    dist2 = cv2.distanceTransform(mask2, cv2.DIST_L2, 3)

    # Normalize weights
    total = dist1 + dist2 + 1e-10
    weight1 = np.repeat((dist1 / total)[:, :, None], 3, axis=2)
    weight2 = np.repeat((dist2 / total)[:, :, None], 3, axis=2)

    blended = weight1 * canvas1 + weight2 * canvas2
    return blended.astype(np.uint8)

## Part 9 — Final Blended Panorama

In [ ]:
# Blend LEFT + CENTER, then blend result with RIGHT
temp = blend_images(naive_left, naive)
final_panorama = blend_images(temp, naive_right)

# Crop black borders
gray_final = cv2.cvtColor(final_panorama, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray_final, 1, 255, cv2.THRESH_BINARY)
coords = cv2.findNonZero(thresh)

if coords is not None:
    x, y, w, h = cv2.boundingRect(coords)
    final_cropped = final_panorama[y:y+h, x:x+w]
else:
    print("Warning: Cropping skipped")
    final_cropped = final_panorama

plt.figure(figsize=(16,7))

plt.subplot(1,2,1)
plt.imshow(cv2.cvtColor(naive, cv2.COLOR_BGR2RGB))
plt.title("Naive Stitch (Before Blending)")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(cv2.cvtColor(final_cropped, cv2.COLOR_BGR2RGB))
plt.title("Final Panorama (After Seamless Blending)")
plt.axis("off")

plt.show()

# Save result
cv2.imwrite("panorama_result.png", final_cropped)

## Part 10 — Manual Crop Viewer

Interactive sliders to trim remaining black borders (needs `ipywidgets`; works in Colab/Jupyter).

In [ ]:
from ipywidgets import interact, IntSlider

img = final_panorama
h, w = img.shape[:2]

def crop_view(top=0, bottom=h, left=0, right=w):
    cropped = img[top:bottom, left:right]
    plt.figure(figsize=(10,5))
    plt.imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

interact(
    crop_view,
    top=IntSlider(0, 0, h-1, 1),
    bottom=IntSlider(h, 1, h, 1),
    left=IntSlider(0, 0, w-1, 1),
    right=IntSlider(w, 1, w, 1),
)